In [1]:
# Cel 1: Imports en Data laden
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

# Laden van de dataset
try:
    df = pd.read_csv('equipment_failure_data_1.csv')
    print(f"Data Loaded. Total rows: {len(df)}")
except:
    print("Data not found")

# Definieer kolommen
TARGET = 'EQUIPMENT_FAILURE'
REGION_COL = 'REGION_CLUSTER'
FEATURES = ['S15', 'S17', 'S13', 'S5', 'S16', 'S19', 'S18', 'S8', 'AGE_OF_EQUIPMENT']

df = df.dropna(subset=FEATURES + [TARGET])

Data Loaded. Total rows: 149855


In [ ]:
# Cel 2: Sensor Analyse (Wat komt er uit de sensoren?)
print("Gemiddelde sensorwaarden bij wel/geen falen:")
analysis = df.groupby(TARGET)[FEATURES].mean()
display(analysis)

# Visualisatie van de belangrijkste sensorverschillen
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x=TARGET, y='S15')
plt.title('Verdeling van Sensor S15 bij Falen vs. Geen Falen')
plt.show()

In [ ]:
# Cel 3: De Experimenten (Local vs Foreign)
def run_experiment(train_df, test_df, title):
    X_train, y_train = train_df[FEATURES], train_df[TARGET]
    X_test, y_test = test_df[FEATURES], test_df[TARGET]
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    print(f"\nResultaten voor {title}:")
    print(f"Recall: {recall_score(y_test, preds):.4f}")
    print(f"F1-Score: {f1_score(y_test, preds):.4f}")
    return model

regions = df[REGION_COL].unique()
reg_a, reg_b = regions[0], regions[1]

# Local
data_a = df[df[REGION_COL] == reg_a]
train_l, test_l = train_test_split(data_a, test_size=0.3, random_state=42)
model_l = run_experiment(train_l, test_l, f"LOCAL ({reg_a})")

# Foreign
data_b = df[df[REGION_COL] == reg_b]
model_f = run_experiment(data_a, data_b, f"FOREIGN ({reg_a} naar {reg_b})")